In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import requests
import warnings
warnings.filterwarnings("ignore")

print("OK")

OK


In [2]:
# Gerresheimer vs peers del sector pharma packaging
COMPANIES = {
    "Gerresheimer":  "GXI.DE",
    "Schott Pharma": "1SXP.DE",
    "Stevanato":     "STVN",
    "AptarGroup":    "ATR",
}

YEAR_START = "2019-01-01"
YEAR_END   = "2024-01-01"

prices_list = []

for name, ticker in COMPANIES.items():
    print(f"Descargando: {name} ({ticker})...")
    df = yf.download(ticker, start=YEAR_START, end=YEAR_END,
                     auto_adjust=True, progress=False)

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    temp = df[["Close"]].copy().reset_index()
    temp.columns = ["date", "close"]
    temp["company"] = name
    temp["ticker"]  = ticker
    prices_list.append(temp)

prices = pd.concat(prices_list, ignore_index=True)
print(f"\nPrecios descargados: {len(prices)} filas")
print(prices.groupby("company")["close"].count())

Descargando: Gerresheimer (GXI.DE)...
Descargando: Schott Pharma (1SXP.DE)...
Descargando: Stevanato (STVN)...
Descargando: AptarGroup (ATR)...

Precios descargados: 3214 filas
company
AptarGroup       1258
Gerresheimer     1272
Schott Pharma      65
Stevanato         619
Name: close, dtype: int64


In [3]:
financials = {}

for name, ticker in COMPANIES.items():
    print(f"Descargando financials: {name}...")
    stock = yf.Ticker(ticker)
    info  = stock.info

    financials[name] = {
        "market_cap":       info.get("marketCap",           None),
        "enterprise_value": info.get("enterpriseValue",     None),
        "revenue_ttm":      info.get("totalRevenue",        None),
        "ebitda":           info.get("ebitda",              None),
        "ev_revenue":       info.get("enterpriseToRevenue", None),
        "ev_ebitda":        info.get("enterpriseToEbitda",  None),
        "pe_ratio":         info.get("trailingPE",          None),
        "gross_margins":    info.get("grossMargins",        None),
        "revenue_growth":   info.get("revenueGrowth",       None),
        "debt_to_equity":   info.get("debtToEquity",        None),
    }

fin_df = pd.DataFrame(financials).T.reset_index()
fin_df.columns = ["company"] + list(fin_df.columns[1:])

print("\nMétricas financieras:")
print(fin_df[["company", "ev_revenue", "ev_ebitda", "pe_ratio", "gross_margins"]].to_string(index=False))

Descargando financials: Gerresheimer...
Descargando financials: Schott Pharma...
Descargando financials: Stevanato...
Descargando financials: AptarGroup...

Métricas financieras:
      company  ev_revenue  ev_ebitda  pe_ratio  gross_margins
 Gerresheimer       1.346      8.721 41.382350        0.23624
Schott Pharma       2.928     11.329 18.357895        0.32936
    Stevanato       4.592     19.147 32.758620        0.29219
   AptarGroup       2.416     11.599 21.743150        0.36672


In [4]:
# Datos de Annual Reports 2019-2023 (EUR millones)
gxl_historical = pd.DataFrame({
    "year":          [2019,   2020,   2021,   2022,   2023],
    "revenue":       [1351.0, 1384.5, 1510.0, 1820.0, 1969.0],
    "ebitda_adj":    [280.0,  305.0,  342.0,  410.0,  498.0],
    "ebit_adj":      [175.0,  195.0,  220.0,  265.0,  320.0],
    "net_income":    [88.0,   95.0,   115.0,  138.0,  158.0],
    "capex":         [145.0,  162.0,  195.0,  230.0,  285.0],
    "net_debt":      [980.0,  950.0,  1100.0, 1350.0, 1420.0],
})

gxl_historical["ebitda_margin"] = (gxl_historical["ebitda_adj"] / gxl_historical["revenue"] * 100).round(1)
gxl_historical["revenue_growth"] = gxl_historical["revenue"].pct_change() * 100
gxl_historical["revenue_growth"] = gxl_historical["revenue_growth"].round(1)

print("Gerresheimer Historical Financials (EUR mn):")
print(gxl_historical.to_string(index=False))

Gerresheimer Historical Financials (EUR mn):
 year  revenue  ebitda_adj  ebit_adj  net_income  capex  net_debt  ebitda_margin  revenue_growth
 2019   1351.0       280.0     175.0        88.0  145.0     980.0           20.7             NaN
 2020   1384.5       305.0     195.0        95.0  162.0     950.0           22.0             2.5
 2021   1510.0       342.0     220.0       115.0  195.0    1100.0           22.6             9.1
 2022   1820.0       410.0     265.0       138.0  230.0    1350.0           22.5            20.5
 2023   1969.0       498.0     320.0       158.0  285.0    1420.0           25.3             8.2


In [5]:
prices.to_csv("../data/prices.csv", index=False)
fin_df.to_csv("../data/financials.csv", index=False)
gxl_historical.to_csv("../data/gxl_historical.csv", index=False)

print("Guardado OK")
print(f"\nShape prices: {prices.shape}")
print(f"Shape financials: {fin_df.shape}")
print(f"Shape historical: {gxl_historical.shape}")

Guardado OK

Shape prices: (3214, 4)
Shape financials: (4, 11)
Shape historical: (5, 9)
